# Evalaute Agent over Benchmark 

Run an agent over benchmark JSONL files and write predicitons.

1. Too add a new agents, add import in the `load_agent` function. 
2. Add list of benchmarks to test against in `args` (arguments)

In [ ]:
# Global variables and paths relative to the script location; change root (and others) if moving the location of the script.

import argparse
import datetime
from datetime import datetime
from typing import Optional, Any, Iterable
import numpy as np
from dataclasses import dataclass, field
import os
import json
from pathlib import Path
import re
import string
import unicodedata

SEED = 42

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "experiments"
BENCHMARKS_DIR = ROOT / "benchmarks"
LOG_DIR = ROOT / "logs"

RAW_DIR = DATA_DIR / "raw"
SAMPLES_DIR = DATA_DIR / "samples"
HF_DATASETS_CACHE = DATA_DIR / "hf_cache"
FIGURES_DIR = DATA_DIR / "figures"

for d in [
    DATA_DIR,
    OUTPUT_DIR,
    BENCHMARKS_DIR,
    SAMPLES_DIR,
    LOG_DIR,
    FIGURES_DIR,
    RAW_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# set default HF cache dir to data/hf_cache
# os.environ.setdefault("hf_datasets_cache", str(HF_DATASETS_CACHE))

# testing a potential "arguments" structure for the main script;
# arguments
parser = argparse.ArgumentParser()
parser.add_argument("--epochs", type=int, default=5)
parser.add_argument("--lr", type=float, default=0.001)
parser.add_argument("--model_name", type=str, default="bert-base-uncased")
parser.add_argument("--agent", type=str, default="direct", help="Name of the agent to run")
parser.add_argument(
    "--benchmarks",
    type=str,
    nargs="+",
    default=[
        "hotpotqa.sample",
    ],
    help="List of benchmarks to run the agent on",
)

# REMOVE THIS LATER - only for ipynb testing
args = parser.parse_args(args=[])

# file paths
benchmarks_files = [BENCHMARKS_DIR / f"{b}.jsonl" for b in args.benchmarks]
# TODO: check if files exist
# Add cleaner check and convert to filepath objects
benchmarks_files = [Path(f) for f in benchmarks_files]
for f in benchmarks_files:
    if not f.exists():
        raise FileNotFoundError(f"File not found: {f}")
    print(f"Found benchmark file: {f}")

Found benchmark file: /Users/dalmia/Developer/maintenance-agent/benchmarks/hotpotqa.sample.jsonl


In [2]:
# Load agent
def load_agent(name: str, **kwargs):
    """Lazy-import and instantiate an agent by name.

    Add a new branch when you wirer a new agent under src/agents/. Keeping these imports lazy means you don't pay the cost of importing transformers / litellm / langchain unless you actually use them."""

    if name == "mock":
        from src.agents.mock_agent import MockAgent

        return MockAgent(**kwargs)

    elif name == "direct":
        from src.agents.direct_agent import DirectAgent
        from src.llm.transformers_backend import TransformersLLMBackend

        # The transformers backend has lazy loading enable so this step should be fast for now
        model_id = kwargs.get("model_id", "meta-llama/Llama-3.2-3B-Instruct")
        llm = TransformersLLMBackend(model_id=model_id)
        return DirectAgent(model_id=model_id, llm=llm, **kwargs)

    raise ValueError(f"Unknown agent name: {name}")


In [3]:
# helpers
def read_jsonl(path: Path) -> list[dict]:
    """Read a jsonl file and return a list of dicts."""
    out = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


# append jsonl to a file
def append_jsonl(path: Path, record: dict) -> None:
    """Append a dict as a jsonl record to a file."""
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def _coerce_result(out: Any) -> tuple[str, str, list, dict]:
    """Normalize whatever the agent returned into (answer, raw, trace, meta)."""
    # TODO make it compatible with TemplateResults and TemplateStep
    if hasattr(out, "answer"):
        return (
            str(getattr(out, "answer") or ""),
            str(getattr(out, "raw_output", "") or ""),
            list(getattr(out, "trace", []) or []),
            dict(getattr(out, "metadata", {}) or {}),
        )
    if isinstance(out, dict):
        return (
            str(out.get("answer", "")),
            str(out.get("raw_output", "")),
            list(out.get("trace", []) or []),
            dict(out.get("metadata", {}) or {}),
        )
    if isinstance(out, str):
        return (out, out, [], {})
    return (str(out), str(out), [], {})


def run_agent_on_one_example(agent, ex: dict, agent_name: str) -> dict:
    """Run the agent on one example and return the result as a dict."""

    answer = raw = ""
    trace: list = []
    meta_out: dict = {}
    error: Optional[str] = None

    # TODO add timing, token usage, and other metadata about the agent's run
    try:
        out = agent.run(ex["question"])
        answer, raw, trace, meta_out = _coerce_result(out)
    except KeyboardInterrupt:
        raise
    except Exception as e:
        error = f"{type(e).__name__}: {str(e)}"

    return {
        "id": ex["id"],
        "benchmark": ex.get("benchmark"),
        "question": ex["question"],
        "answer": ex.get("answer"),  # gold answer if available
        "aliases": ex.get("aliases", []),
        "meta": ex.get("meta", {}),  # any additional metadata from the example
        # Prediction results
        "prediction": answer,
        "raw_output": raw,
        "trace": trace,
        "agent": agent_name,
        # TODO add latency, token usage,
        "error": error,
        "agent_metadata": meta_out,
    }

Design Notes 

* SQuAD-style answer normalization - lowercase, drop articles, drop punctuation, collapse whitespace. 
* Token-level F1 max-over-aliases (TriviaQA, Natural Questions)

In [ ]:
_ARTICLES_RE = re.compile(r"\b(a|an|the)\b", flags=re.IGNORECASE)
_PUNCT_TABLE = str.maketrans("", "", string.punctuation)

# Heuristic abstention detection. Conservative — we only flag obviously
# non-attempts. Anything else counts as an attempt, even if wrong.
_ABSTAIN_PATTERNS = [
    r"^\s*$",
    r"^i (don'?t|do not) know",
    r"^i'?m not sure",
    r"^unknown\b",
    r"^n/?a\b",
    r"^cannot (be )?(determined|answered)",
    r"^insufficient (information|context)",
]
_ABSTAIN_RE = re.compile("|".join(_ABSTAIN_PATTERNS), flags=re.IGNORECASE)


def normalize_answer(s: Optional[str]) -> str:
    """SQuAD-2016 normalization: NFKC, lowercase, strip articles, strip
    punctuation, collapse whitespace. Returns "" for None.
    """
    if s is None:
        return ""
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = _ARTICLES_RE.sub(" ", s)
    s = s.translate(_PUNCT_TABLE)
    return " ".join(s.split())


def _gold_list(record: dict) -> list[str]:
    """Collect non-empty gold variants: primray answer + aliases."""
    golds = []
    a = record.get("answer")
    if a:
        golds.append(str(a))
    golds.extend(str(x) for x in (record.get("aliases") or []) if x)
    return golds


def exact_match(pred: str, golds: list[str]) -> float:
    if not golds:
        return 0.0
    p = normalize_answer(pred)
    return float(any(p == normalize_answer(g) for g in golds))


def token_f1(pred: str, golds: list[str]) -> float:
    """Max over aliases of pairwise token-overlap F1."""
    if not golds:
        return 0.0

    def _f1(p: str, g: str) -> float:
        p_tokens = set(normalize_answer(p).split())
        g_tokens = set(normalize_answer(g).split())
        if not p_tokens and not g_tokens:
            return 1.0
        if not p_tokens or not g_tokens:
            return 0.0
        precision = len(p_tokens & g_tokens) / len(p_tokens)
        recall = len(p_tokens & g_tokens) / len(g_tokens)
        if precision + recall == 0:
            return 0.0
        return 2 * precision * recall / (precision + recall)

    return max(_f1(pred, g) for g in golds)


def substring_match(pred: str, golds: list[str]) -> float:
    """1.0 if any normalized gold appears as a substring of the normalized prediction. USeful when the model wraps the answer in surrounding text ("The answer is Paris.") - common with chat-tuned LMs."""
    if not golds:
        return 0.0
    p = normalize_answer(pred)
    if not p:
        return 0.0
    return float(any(normalize_answer(g) and normalize_answer(g) in p for g in golds))


def is_attempt(pred: str) -> bool:
    return _ABSTAIN_RE.match(pred or "") is None


@dataclass
class ScoredRecord:
    id: Any
    benchmark: Optional[str]
    question: str
    gold: Optional[str]
    aliases: list[str]
    prediction: str
    em: float
    f1: float
    substring: float
    attempted: bool
    latency_s: float
    error: Optional[str]
    meta: dict


def score_record(record: dict) -> ScoredRecord:
    """Score a single prediction record (input == one line of predicitons JSONL)"""
    pred = record.get("prediction", "")
    golds = _gold_list(record)
    return ScoredRecord(
        id=record.get("id"),
        benchmark=record.get("benchmark"),
        question=record.get("question", ""),
        gold=record.get("answer"),
        aliases=record.get("aliases", []),
        prediction=pred,
        em=exact_match(pred, golds),
        f1=token_f1(pred, golds),
        substring=substring_match(pred, golds),
        attempted=is_attempt(pred),
        latency_s=float(record.get("latency_s", 0.0)),
        error=record.get("error"),
        meta=dict(record.get("meta", {})),
    )


def aggregate(
    records: Iterable[ScoredRecord],
    n_bootstrap: int = 1000,
    seed: int = 13,
) -> dict:
    """Summarize scored records: means, CIs, attempt rate, errors, latency."""
    records = list(records)
    n = len(records)
    if n == 0:
        return {"n": 0}

    em = np.array([r.em for r in records])
    f1 = np.array([r.f1 for r in records])
    substring = np.array([r.substring for r in records])
    att = np.array([float(r.attempted) for r in records])
    err_count = sum(1 for r in records if r.error)
    attempted_records = [r for r in records if r.attempted]
    em_attempted = np.mean([r.em for r in attempted_records]) if attempted_records else 0.0

    # TODO Add metrics for lanecy, and attempted-latency,
    # TODO add confidence intervals via bootstrapping
    return {
        "n": n,
        "em": {
            "mean": float(em.mean()),
        },
        "f1": {
            "mean": float(f1.mean()),
        },
        "substring": {
            "mean": float(substring.mean()),
        },
        "attempt_rate": float(att.mean()),
        "em_when_attempted": float(em_attempted),
        "n_errors": err_count,
        "error_rate": float(err_count / n),
    }


In [5]:
# main
limit = None
run_id = f"{datetime.now():%Y%m%d_%H%M%S}_{args.agent}"
run_dir = OUTPUT_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=False)

print("Building the agent...")
agent = load_agent(args.agent)


for f in benchmarks_files:
    scored_results = []
    print("Running the agent on the benchmarks...")
    examples = read_jsonl(f)
    total_in_file = len(examples)
    print(f"Total examples in {f.name}: {total_in_file}")

    if limit is not None:
        examples = examples[:limit]
        print(f"Limiting to first {limit} examples for testing.")

    for i, example in enumerate(examples):
        result = run_agent_on_one_example(agent, example, args.agent)
        scored_results.append(score_record(result))
        append_jsonl(run_dir / f"{args.agent}_{f.name}", result)
        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(examples)} examples from {f.name}")

    print(f"Finished running agent on {f.name}. Scoring results...")
    agg = aggregate(scored_results)
    print(f"Aggregate results for {f.name}: {agg}")
    print("Summary - ")
    print(
        f"EM: {agg['em']['mean']:.3f}, F1: {agg['f1']['mean']:.3f}, Attempt Rate: {agg['attempt_rate']:.3f}, EM when attempted: {agg['em_when_attempted']:.3f}, Error Rate: {agg['error_rate']:.3f}"
    )

Building the agent...


/Users/dalmia/Developer/maintenance-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 254/254 [00:00<00:00, 9554.22it/s]
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Running the agent on the benchmarks...
Total examples in hotpotqa.sample.jsonl: 500


ERROR:tornado.general:SEND Error: Host unreachable
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Processed 10/500 examples from hotpotqa.sample.jsonl
Processed 20/500 examples from hotpotqa.sample.jsonl
Processed 30/500 examples from hotpotqa.sample.jsonl
Processed 40/500 examples from hotpotqa.sample.jsonl
Processed 50/500 examples from hotpotqa.sample.jsonl
Processed 60/500 examples from hotpotqa.sample.jsonl
Processed 70/500 examples from hotpotqa.sample.jsonl
Processed 80/500 examples from hotpotqa.sample.jsonl
Processed 90/500 examples from hotpotqa.sample.jsonl
Processed 100/500 examples from hotpotqa.sample.jsonl
Processed 110/500 examples from hotpotqa.sample.jsonl
Processed 120/500 examples from hotpotqa.sample.jsonl
Processed 130/500 examples from hotpotqa.sample.jsonl
Processed 140/500 examples from hotpotqa.sample.jsonl
Processed 150/500 examples from hotpotqa.sample.jsonl
Processed 160/500 examples from hotpotqa.sample.jsonl
Processed 170/500 examples from hotpotqa.sample.jsonl
Processed 180/500 examples from hotpotqa.sample.jsonl
Processed 190/500 examples from hotpo